## Calculate IAA Score for 225 Batch 2 (100 docs each, 10-20% overlap)
The IAA score is based on the 40 docs that we completed with overlap. 

In [41]:
import json
import pandas as pd
from sklearn.metrics import cohen_kappa_score, f1_score

In [43]:
file_path = "220_docs_with_overlap.json"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total tasks in file:", len(data))

Total tasks in file: 220


In [45]:
data[0]

{'id': 260729187,
 'annotations': [{'id': 93272034,
   'completed_by': {'id': 149661,
    'email': 'ysasson@lion.lmu.edu',
    'first_name': '',
    'last_name': ''},
   'result': [{'id': 'JkkKkt37Z2',
     'type': 'labels',
     'value': {'end': 38,
      'text': 'burger king',
      'start': 27,
      'labels': ['Restaurant_Name']},
     'origin': 'manual',
     'to_name': 'text',
     'from_name': 'label'},
    {'id': '309AWIYJMU',
     'type': 'labels',
     'value': {'end': 26, 'text': 'closed', 'start': 20, 'labels': ['Hours']},
     'origin': 'manual',
     'to_name': 'text',
     'from_name': 'label'}],
   'state': 'CREATED',
   'reviews': [],
   'was_cancelled': False,
   'ground_truth': False,
   'created_at': '2026-04-11T21:21:57.428637Z',
   'updated_at': '2026-04-11T21:21:57.428648Z',
   'draft_created_at': '2026-04-11T21:21:43.493670Z',
   'lead_time': 29.322,
   'prediction': {},
   'result_count': 2,
   'unique_id': '9383dbde-081d-461f-9ea0-5dc5cf08320c',
   'import_id'

In [47]:
overlap_tasks = [task for task in data if len(task.get("annotations", [])) >= 2]

print("Tasks with 2+ annotations:", len(overlap_tasks))

Tasks with 2+ annotations: 60


In [49]:
annotators = set()

for task in overlap_tasks:
    for ann in task["annotations"][:2]:
        email = ann.get("completed_by", {}).get("email", "unknown")
        annotators.add(email)

print("Annotators found:")
for a in annotators:
    print("-", a)

Annotators found:
- ysasson@lion.lmu.edu
- bacosta3@lion.lmu.edu


In [51]:
def tokenize_with_spans(text):
    tokens = []
    start_search = 0
    
    for token in text.split():
        start = text.find(token, start_search)
        end = start + len(token)
        tokens.append((token, start, end))
        start_search = end
    
    return tokens

In [53]:
def annotation_to_token_labels(text, annotation_result):
    tokens = tokenize_with_spans(text)
    
    binary_labels = [0] * len(tokens)
    exact_labels = ["O"] * len(tokens)
    
    for res in annotation_result:
        if res.get("type") != "labels":
            continue
        
        value = res.get("value", {})
        start_char = value.get("start")
        end_char = value.get("end")
        labels = value.get("labels", [])
        
        if start_char is None or end_char is None or not labels:
            continue
        
        entity_label = labels[0]
        
        matched_token_indices = []
        
        for i, (token, tok_start, tok_end) in enumerate(tokens):
            overlap = tok_start < end_char and tok_end > start_char
            if overlap:
                matched_token_indices.append(i)
        
        for j, idx in enumerate(matched_token_indices):
            binary_labels[idx] = 1
            if j == 0:
                exact_labels[idx] = f"B-{entity_label}"
            else:
                exact_labels[idx] = f"I-{entity_label}"
    
    token_texts = [t[0] for t in tokens]
    return token_texts, binary_labels, exact_labels

In [55]:
paired_tasks = []

for task in overlap_tasks:
    text = task["data"]["text"]
    
    ann_a = task["annotations"][0]
    ann_b = task["annotations"][1]
    
    email_a = ann_a.get("completed_by", {}).get("email", "annotator_a")
    email_b = ann_b.get("completed_by", {}).get("email", "annotator_b")
    
    result_a = ann_a.get("result", [])
    result_b = ann_b.get("result", [])
    
    tokens_a, binary_a, exact_a = annotation_to_token_labels(text, result_a)
    tokens_b, binary_b, exact_b = annotation_to_token_labels(text, result_b)
    
    if tokens_a == tokens_b:
        paired_tasks.append({
            "text": text,
            "annotator_a": email_a,
            "annotator_b": email_b,
            "tokens": tokens_a,
            "binary_a": binary_a,
            "binary_b": binary_b,
            "exact_a": exact_a,
            "exact_b": exact_b
        })
    else:
        print("Token mismatch found for text:")
        print(text)

print("Valid paired tasks:", len(paired_tasks))

Valid paired tasks: 60


In [57]:
all_binary_a = []
all_binary_b = []
all_exact_a = []
all_exact_b = []

for task in paired_tasks:
    all_binary_a.extend(task["binary_a"])
    all_binary_b.extend(task["binary_b"])
    all_exact_a.extend(task["exact_a"])
    all_exact_b.extend(task["exact_b"])

print("Total tokens compared:", len(all_binary_a))

Total tokens compared: 578


In [59]:
kappa_binary = cohen_kappa_score(all_binary_a, all_binary_b)
f1_binary = f1_score(all_binary_a, all_binary_b)

print("Binary token-level agreement")
print("Cohen's Kappa:", round(kappa_binary, 4))
print("F1 Score:", round(f1_binary, 4))

Binary token-level agreement
Cohen's Kappa: 0.9485
F1 Score: 0.9651


In [63]:
exact_matches = sum(1 for a, b in zip(all_exact_a, all_exact_b) if a == b)
raw_agreement = exact_matches / len(all_exact_a)

print("Raw exact agreement:", round(raw_agreement * 100, 2), "%")

Raw exact agreement: 96.54 %


In [65]:
disagreements = []

for task in paired_tasks:
    for token, a, b in zip(task["tokens"], task["exact_a"], task["exact_b"]):
        if a != b:
            disagreements.append({
                "text": task["text"],
                "token": token,
                "annotator_a": a,
                "annotator_b": b
            })

disagreement_df = pd.DataFrame(disagreements)

print("Total token disagreements:", len(disagreement_df))
disagreement_df.head(20)

Total token disagreements: 20


,text,token,annotator_a,annotator_b
0,can you find me the nearest pizzeria,pizzeria,B-Dish,B-Cuisine
1,can you find some reviews on the new restauran...,reviews,B-Rating,B-Amenity
2,can you find the nearest italian restaurant,nearest,O,B-Location
3,can you find the restaurant carinos italian gr...,italian,B-Cuisine,I-Restaurant_Name
4,can you set up a reservation for the local fre...,bistro,O,B-Amenity
5,can you show me all the places around here tha...,around,B-Location,O
6,can you show me all the places around here tha...,here,I-Location,O
7,can you show me all the places around here tha...,dinner,O,B-Amenity
8,can you show me the lunch menu for buckram sta...,lunch,O,B-Amenity
9,can you show me where i should go for lunch,lunch,O,B-Amenity
